In [11]:
import numpy as np
import pandas as pd
import pickle
import json
import logging
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score, recall_score, roc_auc_score, f1_score
import string
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

In [2]:
import dagshub
import mlflow


dagshub.init(repo_owner='rajeshxdatascience', repo_name='mlops-mini-project', mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/rajeshxdatascience/mlops-mini-project.mlflow")

Accessing as rajeshxdatascience

Initialized MLflow to track repo "rajeshxdatascience/mlops-mini-project"

Repository rajeshxdatascience/mlops-mini-project initialized!

In [3]:
df = pd.read_csv("https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv").drop(columns=['tweet_id'])

In [4]:
# Transform the data

import re
import pandas as pd
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required NLTK resources
nltk.download("stopwords")
nltk.download("wordnet")


def lemmatization(text: str) -> str:
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)


def remove_stop_words(text: str) -> str:
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)


def removing_numbers(text: str) -> str:
    text = " ".join(
        [word for word in text.split() if not word.isdigit()]
    )
    return text


def lower_case(text: str) -> str:
    return text.lower()


def removing_punctuation(text: str) -> str:
    text = re.sub(
        '[%s]' % re.escape(r"""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""),
        " ",
        text
    )
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def removing_urls(text: str) -> str:
    url_pattern = re.compile(r"https?://\S+|www\.\S+")
    return url_pattern.sub("", text)


def normalize_text(df: pd.DataFrame) -> pd.DataFrame:
    df["content"] = df["content"].astype(str)

    df["content"] = df["content"].apply(lower_case)
    df["content"] = df["content"].apply(removing_urls)
    df["content"] = df["content"].apply(removing_numbers)
    df["content"] = df["content"].apply(removing_punctuation)
    df["content"] = df["content"].apply(remove_stop_words)
    df["content"] = df["content"].apply(lemmatization)

    return df


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rajes\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\rajes\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [5]:
df = normalize_text(df)
df.head()

,sentiment,content
0,empty,tiffanylue know listenin bad habit earlier sta...
1,sadness,layin n bed headache ughhhh waitin call
2,sadness,funeral ceremony gloomy friday
3,enthusiasm,want hang friend soon
4,neutral,dannycastillo want trade someone houston ticke...


In [6]:
df['sentiment'].value_counts()

sentiment
neutral       8638
worry         8459
happiness     5209
sadness       5165
love          3842
surprise      2187
fun           1776
relief        1526
hate          1323
empty          827
enthusiasm     759
boredom        179
anger          110
Name: count, dtype: int64

In [7]:
x = df['sentiment'].isin(['happiness','sadness'])
df = df[x]

In [8]:
df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})

C:\Users\rajes\AppData\Local\Temp\ipykernel_6196\1508851655.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})


In [ ]:
# Set the experiment name
mlflow.set_experiment("BoW vs tifidf")

In [9]:
vectorizers = {
    'BoW':CountVectorizer(),
    'TF-IDF':TfidfVectorizer()
}

In [12]:
algorithms = {
    'LogisticRegression': LogisticRegression(),
    'MultinomialNB': MultinomialNB(),
    'XGBoost': XGBClassifier(),
    'RandomForest':RandomForestClassifier(),
    'GradientBoosting':GradientBoostingClassifier()
}

In [17]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# Start the parent run
with mlflow.start_run(run_name="All Experiments") as parent_run:

    # Loop through algorithms
    for algo_name, model in algorithms.items():

        # Loop through feature extraction methods
        for vec_name, vectorizer in vectorizers.items():

            # Start child run
            with mlflow.start_run(
                run_name=f"{algo_name} with {vec_name}",
                nested=True
            ) as child_run:

                # -----------------------------------
                # Feature extraction
                # -----------------------------------
                X = vectorizer.fit_transform(df["content"])
                y = df["sentiment"]

                # -----------------------------------
                # Train-test split
                # -----------------------------------
                X_train, X_test, y_train, y_test = train_test_split(
                    X,
                    y,
                    test_size=0.2,
                    random_state=42
                )

                # -----------------------------------
                # Log preprocessing parameters
                # -----------------------------------
                mlflow.log_param("vectorizer", vec_name)
                mlflow.log_param("algorithm", algo_name)
                mlflow.log_param("test_size", 0.2)

                # -----------------------------------
                # Log model parameters
                # -----------------------------------
                if algo_name == "LogisticRegression":
                    mlflow.log_param("C", model.C)

                elif algo_name == "MultinomialNB":
                    mlflow.log_param("alpha", model.alpha)

                elif algo_name == "XGBoost":
                    mlflow.log_param(
                        "n_estimators",
                        model.n_estimators
                    )
                    mlflow.log_param(
                        "learning_rate",
                        model.learning_rate
                    )

                elif algo_name == "RandomForest":
                    mlflow.log_param(
                        "n_estimators",
                        model.n_estimators
                    )
                    mlflow.log_param(
                        "max_depth",
                        model.max_depth
                    )

                elif algo_name == "GradientBoostingClassifier":
                    mlflow.log_param(
                        "n_estimators",
                        model.n_estimators
                    )
                    mlflow.log_param(
                        "learning_rate",
                        model.learning_rate
                    )
                    mlflow.log_param(
                        "max_depth",
                        model.max_depth
                    )

                # -----------------------------------
                # Train model
                # -----------------------------------
                model.fit(X_train, y_train)

                # -----------------------------------
                # Make predictions
                # -----------------------------------
                y_pred = model.predict(X_test)

                # -----------------------------------
                # Model evaluation
                # -----------------------------------
                accuracy = accuracy_score(
                    y_test,
                    y_pred
                )

                precision = precision_score(
                    y_test,
                    y_pred,
                    zero_division=0
                )

                recall = recall_score(
                    y_test,
                    y_pred,
                    zero_division=0
                )

                f1 = f1_score(
                    y_test,
                    y_pred,
                    zero_division=0
                )

                # -----------------------------------
                # Log evaluation metrics
                # -----------------------------------
                mlflow.log_metric(
                    "accuracy",
                    accuracy
                )

                mlflow.log_metric(
                    "precision",
                    precision
                )

                mlflow.log_metric(
                    "recall",
                    recall
                )

                mlflow.log_metric(
                    "f1",
                    f1
                )

                # -----------------------------------
                # Log model
                # -----------------------------------
                if algo_name == "XGBoost":

                    mlflow.xgboost.log_model(
                        model,
                        name="model"
                    )

                else:

                    mlflow.sklearn.log_model(
                        model,
                        name="model"
                    )

                # -----------------------------------
                # Print results
                # -----------------------------------
                print(
                    f"Algorithm: {algo_name}, "
                    f"Feature Engineering: {vec_name}"
                )

                print(f"Accuracy: {accuracy}")
                print(f"Precision: {precision}")
                print(f"Recall: {recall}")
                print(f"F1 Score: {f1}")
                print("-" * 50)

    # save and log the notebook
    import os
    notebook_path = 'exp1_bow_vs_tfidf.ipynb'
    os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
    mlflow.log_artifact(notebook_path)

Algorithm: LogisticRegression, Feature Engineering: BoW
Accuracy: 0.7980722891566265
Precision: 0.7882011605415861
Recall: 0.8029556650246306
F1 Score: 0.7955100048804294
--------------------------------------------------
🏃 View run LogisticRegression with BoW at: https://dagshub.com/rajeshxdatascience/mlops-mini-project.mlflow/#/experiments/0/runs/caa55f6ae791477080c9169fa0f70021
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/mlops-mini-project.mlflow/#/experiments/0
Algorithm: LogisticRegression, Feature Engineering: TF-IDF
Accuracy: 0.7956626506024096
Precision: 0.7806267806267806
Recall: 0.8098522167487685
F1 Score: 0.7949709864603481
--------------------------------------------------
🏃 View run LogisticRegression with TF-IDF at: https://dagshub.com/rajeshxdatascience/mlops-mini-project.mlflow/#/experiments/0/runs/9a1bf7c9477d45a89e50d72a3b5ae2b4
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/mlops-mini-project.mlflow/#/experiments/0
Algorithm: Multinomi